# MIMIC AKI Outcome Prediction with Trajectory Features
Feature configurations: baseline, summary stats, trajectory probs, and combinations.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import os
import sys
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))
from notebook_utils import biomarker_summary_stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

## Load Data

In [ ]:
pred_with_probs_path = '../../../results/mimic/aki/aki_prediction_dataset_with_probs.csv'
outcome_path = '../../../results/mimic/aki/aki_outcomes.csv'
ts_path = '../../../results/mimic/aki/creatinine_timeseries.csv'

df = pd.read_csv(pred_with_probs_path)
outcomes = pd.read_csv(outcome_path)
creatinine_ts = pd.read_csv(ts_path)

print(f'✓ Loaded prediction dataset with probs: {len(df):,} rows')
print(f'✓ Outcomes: {len(outcomes):,} rows')
print(f'✓ Creatinine TS: {len(creatinine_ts):,} rows')

df = df.merge(outcomes[['hadm_id', 'time_day', 'target_aki_stage3']], on=['hadm_id', 'time_day'], how='left')
df = df[df['target_aki_stage3'].notna()].copy()
df['target_aki_stage3'] = df['target_aki_stage3'].astype(int)

traj_cols = [c for c in df.columns if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]

print(f'Final dataset rows: {len(df):,}')
print(f'Outcome rate: {df["target_aki_stage3"].mean():.1%}')
print(f'Trajectory columns: {len(traj_cols)}')

## Feature Engineering

In [ ]:
summary_7d = biomarker_summary_stats(creatinine_ts, 'creatinine', lookback_days=7)
df = df.merge(summary_7d, on=['hadm_id', 'time_day'], how='left')

exclude_cols = {'hadm_id', 'time_day', 'charttime', 'admittime', 'dischtime', 'target_aki_stage3'}
numeric_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
traj_cols = [c for c in numeric_cols if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]
summary_cols = [c for c in numeric_cols if c.endswith('_7d') or '_trend_' in c or '_change_' in c]
base_cols = [c for c in numeric_cols if c not in traj_cols and c not in summary_cols]

static_name_tokens = ['age', 'gender', 'sex', 'baseline', 'admit', 'admission', 'ethnicity', 'race', 'height', 'weight', 'bmi']
static_cols = [c for c in base_cols if any(tok in c.lower() for tok in static_name_tokens)]
dynamic_cols = [c for c in base_cols if c not in static_cols]

feature_sets = {
    'Trajectory Only': traj_cols,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols + summary_cols + static_cols,
}

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = static_cols + dynamic_cols
    feature_sets['Trajectory + Static + Dynamic'] = traj_cols + static_cols + dynamic_cols
    feature_sets['Summary + Static + Dynamic'] = summary_cols + static_cols + dynamic_cols
    feature_sets['Trajectory + Summary + Static + Dynamic'] = traj_cols + summary_cols + static_cols + dynamic_cols

for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} features')

## Model Comparison

In [ ]:
models_to_evaluate = {
    'LogReg': lambda: LogisticRegression(max_iter=300, n_jobs=-1),
    'RF': lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    'HGB': lambda: HistGradientBoostingClassifier(random_state=42),
    'XGB': lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, eval_metric='logloss'),
}

n_repeats = 10
n_splits = 5
results = {model_name: {} for model_name in models_to_evaluate.keys()}
y = df['target_aki_stage3']
groups = df['hadm_id']

for model_name, model_fn in models_to_evaluate.items():
    dataset_model = df.copy()
    if len(traj_cols) > 0:
        dataset_model[traj_cols] = dataset_model.groupby('hadm_id')[traj_cols].ffill(limit=2)
    if model_name not in ['XGB', 'HGB']:
        if len(traj_cols) > 0:
            dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        if len(summary_cols) > 0:
            dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            continue
        fold_metrics = {'roc_auc': [], 'avg_precision': [], 'y_true': [], 'y_pred': []}
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_splits)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGB', 'HGB']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                model = model_fn()
                model.fit(X_train_scaled, y_train)

                if hasattr(model, 'predict_proba'):
                    probs = model.predict_proba(X_test_scaled)[:, 1]
                else:
                    probs = model.decision_function(X_test_scaled)

                fold_metrics['roc_auc'].append(roc_auc_score(y_test, probs))
                fold_metrics['avg_precision'].append(average_precision_score(y_test, probs))
                fold_metrics['y_true'].extend(y_test.tolist())
                fold_metrics['y_pred'].extend(probs.tolist())

        results[model_name][feature_set_name] = fold_metrics
        print(f"{model_name} | {feature_set_name}: AUROC {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}, AUPRC {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}")

summary_rows = []
for model_name, model_results in results.items():
    for feature_set_name, metrics in model_results.items():
        summary_rows.append({
            'model': model_name,
            'feature_set': feature_set_name,
            'auroc_mean': np.mean(metrics['roc_auc']),
            'auroc_std': np.std(metrics['roc_auc']),
            'auprc_mean': np.mean(metrics['avg_precision']),
            'auprc_std': np.std(metrics['avg_precision'])
        })

results_df = pd.DataFrame(summary_rows)
results_df

## Compare Model Performance

In [ ]:
print("=" * 100)
print("MODEL COMPARISON - All Feature Sets")
print("=" * 100)

summary_df = results_df.copy()
summary_df['ROC-AUC'] = summary_df.apply(lambda r: f"{r.auroc_mean:.3f} ± {r.auroc_std:.3f}", axis=1)
summary_df['AUPR'] = summary_df.apply(lambda r: f"{r.auprc_mean:.3f} ± {r.auprc_std:.3f}", axis=1)

for model_name in summary_df['model'].unique():
    print(f"{model_name}:")
    model_df = summary_df[summary_df['model'] == model_name]
    print(model_df[['feature_set', 'ROC-AUC', 'AUPR']].to_string(index=False))
    print("\n")

print("\n" + "=" * 100)
print("TOP 10 CONFIGURATIONS BY AUROC")
print("=" * 100)
top10 = summary_df.sort_values('auroc_mean', ascending=False).head(10)
print(top10[['model', 'feature_set', 'ROC-AUC', 'AUPR']].to_string(index=False))

print("\n" + "=" * 100)
print("BEST MODEL PER FEATURE SET")
print("=" * 100)
for feature_set_name in summary_df['feature_set'].unique():
    best = summary_df[summary_df['feature_set'] == feature_set_name].sort_values('auroc_mean', ascending=False).iloc[0]
    print(f"  {feature_set_name:45s}: {best['model']:10s} (AUROC={best['auroc_mean']:.3f}, AUPR={best['auprc_mean']:.3f})")

## Visualize Results

In [ ]:
# ROC and PR curves for each model (key feature sets)
for model_name in results.keys():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f'{model_name} - Key Feature Sets Comparison', fontsize=14, fontweight='bold')

    key_sets = [
        'Trajectory Only',
        'Summary Stats Only',
        'Trajectory + Summary Stats',
        'Static + Dynamic',
        'Trajectory + Summary Stats + Static + Dynamic'
    ]

    colors = plt.cm.tab10(np.linspace(0, 1, len(key_sets)))

    for idx, name in enumerate(key_sets):
        if name not in results[model_name]:
            continue
        metrics = results[model_name][name]
        fpr, tpr, _ = roc_curve(metrics['y_true'], metrics['y_pred'])
        auc = np.mean(metrics['roc_auc'])
        ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colors[idx], linewidth=2)

        precision, recall, _ = precision_recall_curve(metrics['y_true'], metrics['y_pred'])
        ap = np.mean(metrics['avg_precision'])
        ax2.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=colors[idx], linewidth=2)

    ax1.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.legend(loc='lower right', fontsize=9)
    ax1.grid(True, alpha=0.3)

    baseline = y.mean()
    ax2.axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.3f})', linewidth=1)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Model Comparison Summary (boxplots)
for model_name in results.keys():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'{model_name} - Performance Across All Feature Sets', fontsize=14, fontweight='bold')

    feature_set_names = list(results[model_name].keys())
    roc_data = [results[model_name][name]['roc_auc'] for name in feature_set_names]
    ap_data = [results[model_name][name]['avg_precision'] for name in feature_set_names]

    bp1 = ax1.boxplot(roc_data, labels=feature_set_names, patch_artist=True)
    ax1.set_ylabel('ROC-AUC')
    ax1.set_title('ROC-AUC Across All Feature Sets')
    ax1.set_xticklabels(feature_set_names, rotation=45, ha='right')
    ax1.grid(alpha=0.3, axis='y')

    bp2 = ax2.boxplot(ap_data, labels=feature_set_names, patch_artist=True)
    ax2.set_ylabel('Average Precision')
    ax2.set_title('AUPR Across All Feature Sets')
    ax2.set_xticklabels(feature_set_names, rotation=45, ha='right')
    ax2.grid(alpha=0.3, axis='y')

    for bp in [bp1, bp2]:
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')

    plt.tight_layout()
    plt.show()

# Summary bar plots (AUROC/AUPRC means)
summary_plot_df = results_df.copy()
plt.figure(figsize=(12, 5))
sns.catplot(data=summary_plot_df, x='feature_set', y='auroc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC AKI: AUROC by Feature Set and Model')
plt.ylabel('AUROC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

sns.catplot(data=summary_plot_df, x='feature_set', y='auprc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC AKI: AUPRC by Feature Set and Model')
plt.ylabel('AUPRC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
summary = results_df.pivot_table(index='model', columns='feature_set', values=['auroc_mean', 'auprc_mean'])
summary

In [ ]:
sns.catplot(data=results_df, x='feature_set', y='auroc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC AKI: AUROC by Feature Set and Model')
plt.ylabel('AUROC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

sns.catplot(data=results_df, x='feature_set', y='auprc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC AKI: AUPRC by Feature Set and Model')
plt.ylabel('AUPRC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()